# 신호등 + 라바콘 YOLOv8n 학습 (Colab GPU)
6클래스: tl_red, tl_green, pedestrian, temi, unknown, **cone**. 데이터: `ds_tl5_cone.zip`
(학습 2,142장 = 신호등 1,242 + 콘 900 / 검증 221장 = 신호등 103 + 콘 118 / 시험 93장 = 콘).

**시작 전**: `런타임 → 런타임 유형 변경 → T4 GPU`. 셀을 위에서부터 차례로 실행.

결과는 드라이브 `MyDrive/tl5_runs/` 에 저장 — 끊기면 1~5번을 다시 실행한 뒤 6번을 실행하면 이어서 학습한다.

In [ ]:
# 1. GPU 확인 — 'Tesla T4' 같은 이름이 나와야 한다
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. ultralytics 설치
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# 3. 구글 드라이브 연결 (팝업에서 ds_aug.zip 이 있는 계정으로 허용)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. 데이터셋을 Colab 로컬 디스크로 복사·압축 해제 (드라이브에서 바로 읽으면 느리다)
import os, glob, subprocess
ZIP = '/content/drive/MyDrive/ds_tl5_cone.zip'   # ← 다른 폴더에 있으면 여기만 고친다
if not os.path.exists(ZIP):
    print('없음:', ZIP); print('후보:', glob.glob('/content/drive/MyDrive/**/ds_tl5_cone*.zip', recursive=True)[:5])
else:
    !cp "$ZIP" /content/ds_tl5_cone.zip && unzip -q -o /content/ds_tl5_cone.zip -d /content/
    print('풀림:', os.listdir('/content/ds_tl5_cone'))

In [ ]:
# 5. data.yaml 경로를 Colab 경로로 고치고 개수 확인 + 신호등 전용 검증셋 목록(9월 수치와 비교용)
!sed -i "s|^path: .*|path: /content/ds_tl5_cone|" /content/ds_tl5_cone/data.yaml
!cat /content/ds_tl5_cone/data.yaml
import glob
for sp, want in (('train', 2142), ('val', 221), ('test', 93)):
    n = len(glob.glob(f'/content/ds_tl5_cone/images/{sp}/*.jpg')); m = len(glob.glob(f'/content/ds_tl5_cone/labels/{sp}/*.txt'))
    print(sp, '이미지', n, '라벨', m, '✓' if n == m == want else f'★ 예상 {want}')
tl = sorted(glob.glob('/content/ds_tl5_cone/images/val/tl_*.jpg'))
open('/content/ds_tl5_cone/val_tl_only.txt', 'w').write('\n'.join(tl) + '\n')
y = open('/content/ds_tl5_cone/data.yaml').read().replace('val: images/val', 'val: val_tl_only.txt')
open('/content/ds_tl5_cone/data_tlval.yaml', 'w').write(y); print('신호등 전용 검증', len(tl), '장 (103 이어야 함)')

In [ ]:
# 6. 학습 — 1280 해상도, 최대 100에폭, 20에폭 동안 안 나아지면 조기 종료. T4 기준 대략 1~1.5시간
#    결과는 드라이브에 바로 저장되고, 끊긴 뒤 다시 실행하면 last.pt 에서 이어서 한다
import os
from ultralytics import YOLO
PROJ, NAME = '/content/drive/MyDrive/tl5_runs', 'tl5cone_1280'
last = f'{PROJ}/{NAME}/weights/last.pt'
if os.path.exists(last):
    print('이어서 학습:', last); YOLO(last).train(resume=True)
else:
    YOLO('yolov8n.pt').train(data='/content/ds_tl5_cone/data.yaml', imgsz=1280, epochs=100, patience=20, batch=16,
                             project=PROJ, name=NAME, exist_ok=True)

In [ ]:
# 7. 검증 ① 전체(신호등+콘)  ② 신호등 검증셋 103장만 — 9월 수치와 같은 조건 비교 (가장 중요한 건 tl_green 의 P)
from ultralytics import YOLO
best = '/content/drive/MyDrive/tl5_runs/tl5cone_1280/weights/best.pt'
m = YOLO(best)
def show(r, title):
    b = r.box; print(f'\n== {title}'); print(f"{'':10s} {'P':>6s} {'R':>6s} {'mAP50':>6s} {'50-95':>6s}")
    print(f"{'전체':10s} {b.mp:6.3f} {b.mr:6.3f} {b.map50:6.3f} {b.map:6.3f}")
    for j, i in enumerate(b.ap_class_index): print(f'{m.names[int(i)]:10s} {b.p[j]:6.3f} {b.r[j]:6.3f} {b.ap50[j]:6.3f} {b.ap[j]:6.3f}')
show(m.val(data='/content/ds_tl5_cone/data.yaml', imgsz=1280, plots=True), '전체 검증 221장')
show(m.val(data='/content/ds_tl5_cone/data_tlval.yaml', imgsz=1280, plots=False), '신호등 검증 103장만')
show(m.val(data='/content/ds_tl5_cone/data.yaml', split='test', imgsz=1280, plots=False), '콘 시험 93장')
print('\n비교 기준 (신호등 검증 103장)')
print('9월 1280+증강   전체 P 0.951 R 0.906 mAP50 0.945 mAP50-95 0.681 | tl_green P 0.981 R 0.895')
print('NUC 640 예비    전체 P 0.927 R 0.890 mAP50 0.941 mAP50-95 0.677 | tl_green P 0.876 R 0.789')

In [ ]:
# 8. ONNX 변환 + 결과 정리 (내부 PC TensorRT 8.5 호환: opset 12, 고정 크기)
import shutil, os
from ultralytics import YOLO
d = '/content/drive/MyDrive/tl5_runs/tl5cone_1280'
onnx = YOLO(f'{d}/weights/best.pt').export(format='onnx', imgsz=1280, opset=12, simplify=True, dynamic=False)
out = '/content/drive/MyDrive/tl5cone_result'; os.makedirs(out, exist_ok=True)
shutil.copy(f'{d}/weights/best.pt', f'{out}/tl5cone_1280.pt')
shutil.copy(onnx, f'{out}/tl5cone_1280.onnx')
for f in ('results.csv', 'args.yaml', 'confusion_matrix.png', 'results.png'):
    if os.path.exists(f'{d}/{f}'): shutil.copy(f'{d}/{f}', out)
print('드라이브 MyDrive/tl5cone_result 에 저장:', os.listdir(out))

## 끝나면
`MyDrive/tl5cone_result/` 폴더(.pt, .onnx, results.csv 등)를 받아서 NUC `~/Downloads/` 에 넣는다.
콘 데이터는 Roboflow 공개 데이터(CC BY 4.0, 출처는 `CONE_SOURCE_README.roboflow.txt`). 야외 잔디밭 사진이라 실내 경기장 콘은 따로 확인이 필요하다.